# The Train Speed Problem

In this notebook we are going to solve the following text problem using the constraint solver `Z3`.
<ul>
    <li>A train travels at a uniform speed for 360 miles.</li>
    <li>The train would have taken 48 minutes less to travel the same distance if it had been faster by 5 miles per hour.</li>
    Find the speed of the train!
</ul>

First, we initialize the `z3-solver` library.

In [1]:
import { init } from 'z3-solver';
const { Context } = await init();
const Z3 = Context("main");

As the speed and the time might be fractional numbers, we must declare these variables using `ctx.Real.const` instead of using `Z3.Int.const`.

In [2]:
const time = Z3.Real.const('time');
const speed = Z3.Real.const('speed');

We create the *solver* object.

In [3]:
const S = new Z3.Solver();

Now we formulate our equations and add them to the solver:

* 360 = speed*time
* 360 = (speed+5)*(time-4/5)

In [4]:
S.add(time.mul(speed).eq(360));
S.add(time.sub(0.8).mul(speed.add(5)).eq(360));
S.add(time.ge(0));
S.add(speed.ge(0));

We check if the constraints are satisfiable.

In [5]:
await S.check();

sat


We extract the model to find the specific values for `speed` and `time`.

In [6]:
const model = S.model();

Finally, we extract the numerical values using `eval`. Because we are using real numbers, we convert the Z3 AST objects to strings and parse them using the function `Number()`.

In [7]:
const s = Number(model.eval(speed));
const t = Number(model.eval(time));

In [8]:
console.log(`The speed of the train is ${s} mph.`);
console.log(`The standard time taken is ${t} hours.`);
console.log(`\nVerification:`);
console.log(`Standard: ${s} mph * ${t} hours = ${s * t} miles.`);
console.log(`Faster: (${s} + 5) mph * (${t} - 0.8) hours = ${s + 5} * ${t - 0.8} = ${(s + 5) * (t - 0.8)} miles.`);

The speed of the train is 45 mph.
The standard time taken is 8 hours.

Verification:
Standard: 45 mph * 8 hours = 360 miles.
Faster: (45 + 5) mph * (8 - 0.8) hours = 50 * 7.2 = 360 miles.
